# NER-Arbeitsdateien erzeugen

**Ordnerkontext:** liegt in `00_werkzeuge/`, liest aus `02_quelldaten/`,
schreibt nach `03_annotation_laufend/ner/tranche<N>/`.

## Konfiguration


In [ ]:
from pathlib import Path

BASIS_QUELL = Path("..") / "02_quelldaten"
CONLL       = BASIS_QUELL / "goldstandard_v1.conll"

TRANCHE     = 2                        # 1 oder 2
BEARBEITER  = ["A1", "A3", "A7", "A8"]
MARKIERUNGEN = ["O", "B-FIGUR", "I-FIGUR"]
ROTIEREN    = True
UEBERSCHREIBEN = False

BASIS_ARBEIT = (Path("..") / "03_annotation_laufend" / "ner"
                / f"tranche{TRANCHE}")

assert CONLL.exists(), f"{CONLL} nicht gefunden — aus 00_werkzeuge/ starten?"
print(f"Quelle:  {CONLL}")
print(f"Ziel:    {BASIS_ARBEIT}")
print(f"Tranche: {TRANCHE} | Bearbeitende: {BEARBEITER}")
print(f"Auswahl: {MARKIERUNGEN}")
print("Ueberschreiben:", "JA" if UEBERSCHREIBEN else "nein (Schutz aktiv)")

Quelle:  ../02_quelldaten/goldstandard_v1.conll
Ziel:    ../03_annotation_laufend/ner/tranche2
Tranche: 2 | Bearbeitende: ['A1', 'A3', 'A7', 'A8']
Auswahl: ['O', 'B-FIGUR', 'I-FIGUR']
Ueberschreiben: nein (Schutz aktiv)


## Schritt 1 — Goldstandard einlesen, Wortart- und Markierungsspalte verwerfen


In [2]:
zeilen = CONLL.read_text(encoding="utf-8").splitlines()

bloecke = []
meta, saetze, akt = {}, [], []
lauf = {"token": 0, "satz": 0, "block": 0}

def satz_abschliessen():
    global akt
    if akt:
        lauf["satz"] += 1
        text = " ".join(tk["token"] for tk in akt)
        for nr, tk in enumerate(akt):
            tk["satz"] = lauf["satz"]
            tk["token_nr"] = nr
            tk["satz_text"] = text if nr == 0 else ""
        saetze.append(akt)
        akt = []

def block_abschliessen():
    global meta, saetze
    satz_abschliessen()
    if saetze:
        lauf["block"] += 1
        for s in saetze:
            for tk in s:
                tk["block_nr"] = lauf["block"]
        bloecke.append({"meta": dict(meta), "saetze": saetze})
    saetze = []

for z in zeilen:
    if z.startswith("#"):
        if " = " in z:
            k, w = z[1:].split(" = ", 1)
            if k.strip() == "erzaehlung":
                block_abschliessen()
            meta[k.strip()] = w.strip()
        continue
    if not z.strip():
        satz_abschliessen()
        continue
    #NUR Spalte 1; Spalte 2 (Wortart) und Spalte 3 (Markierung) bleiben aussen vor
    akt.append({"id": lauf["token"], "token": z.split("\t")[0]})
    lauf["token"] += 1
block_abschliessen()

n_token = sum(len(s) for b in bloecke for s in b["saetze"])
print(f"Gelesen: {len(bloecke)} Bloecke, {lauf['satz']} Saetze, {n_token} Token")
print("Uebernommene Spalten je Zeile: 1 (Token).")
print("Die Wortartenspalte wurde nicht ausgelesen — kein Ankereffekt.")

tranche_bloecke = [b for b in bloecke if b["meta"].get("tranche") == str(TRANCHE)]
t_token = sum(len(s) for b in tranche_bloecke for s in b["saetze"])
t_saetze = sum(len(b["saetze"]) for b in tranche_bloecke)
print(f"\nTranche {TRANCHE}: {len(tranche_bloecke)} Bloecke, "
      f"{t_saetze} Saetze, {t_token} Token")
assert tranche_bloecke, f"Keine Bloecke mit tranche = {TRANCHE}."

Gelesen: 40 Bloecke, 150 Saetze, 4137 Token
Uebernommene Spalten je Zeile: 1 (Token).
Die Wortartenspalte wurde nicht ausgelesen — kein Ankereffekt.

Tranche 2: 20 Bloecke, 75 Saetze, 2343 Token


## Schritt 2 — Blockreihenfolge je Person festlegen

In [3]:
reihenfolgen = {}
for i, person in enumerate(BEARBEITER):
    versatz = round(i * len(tranche_bloecke) / len(BEARBEITER)) if ROTIEREN else 0
    reihenfolgen[person] = tranche_bloecke[versatz:] + tranche_bloecke[:versatz]

for person, folge in reihenfolgen.items():
    nummern = [b["saetze"][0][0]["block_nr"] for b in folge]
    print(f"  {person}: beginnt mit Block {nummern[0]:>2} | Folge "
          + " ".join(str(n) for n in nummern[:10]) + " ...")

  A1: beginnt mit Block  1 | Folge 1 4 5 8 10 12 13 16 18 19 ...
  A3: beginnt mit Block 12 | Folge 12 13 16 18 19 22 23 26 28 29 ...
  A7: beginnt mit Block 22 | Folge 22 23 26 28 29 32 34 35 37 39 ...
  A8: beginnt mit Block 32 | Folge 32 34 35 37 39 1 4 5 8 10 ...


## Schritt 3 — Arbeitsmappen bauen


In [4]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Protection
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter

SPALTEN = ["id", "satz", "token_nr", "token", "markierung", "unsicher",
           "kommentar", "satz_text", "sammlung", "erzaehlung"]
BREITEN = {"id": 7, "satz": 7, "token_nr": 9, "token": 20,
           "markierung": 13, "unsicher": 10, "kommentar": 26,
           "satz_text": 70, "sammlung": 12, "erzaehlung": 34}
EDITIERBAR = {"markierung", "unsicher", "kommentar"}


def baue_mappe(person, folge, ziel):
    wb = Workbook()
    ws = wb.active
    ws.title = "NER"

    kopf_font = Font(name="Arial", bold=True, color="FFFFFF")
    kopf_fill = PatternFill("solid", fgColor="404040")
    edit_fill = PatternFill("solid", fgColor="FFF2CC")
    satz_fill = PatternFill("solid", fgColor="F2F2F2")
    zell_font = Font(name="Arial")

    for j, name in enumerate(SPALTEN, start=1):
        c = ws.cell(row=1, column=j, value=name)
        c.font, c.fill = kopf_font, kopf_fill
        c.protection = Protection(locked=True)
        ws.column_dimensions[get_column_letter(j)].width = BREITEN[name]

    zeile = 2
    for block in folge:
        for satz in block["saetze"]:
            streifen = (satz[0]["satz"] % 2 == 0)
            for tk in satz:
                werte = {"id": tk["id"], "satz": tk["satz"],
                         "token_nr": tk["token_nr"], "token": tk["token"],
                         "markierung": "O", "unsicher": "", "kommentar": "",
                         "satz_text": tk["satz_text"],
                         "sammlung": block["meta"].get("sammlung", ""),
                         "erzaehlung": block["meta"].get("erzaehlung", "")}
                for j, name in enumerate(SPALTEN, start=1):
                    c = ws.cell(row=zeile, column=j, value=werte[name])
                    c.font = zell_font
                    if name == "satz_text":
                        c.alignment = Alignment(vertical="top")
                    if name in EDITIERBAR:
                        c.fill = edit_fill
                        c.protection = Protection(locked=False)
                    else:
                        c.protection = Protection(locked=True)
                        if streifen:
                            c.fill = satz_fill
                zeile += 1

    letzte = zeile - 1
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:{get_column_letter(len(SPALTEN))}{letzte}"

    mk = get_column_letter(SPALTEN.index("markierung") + 1)
    dv = DataValidation(type="list",
                        formula1='"' + ",".join(MARKIERUNGEN) + '"',
                        allow_blank=False, showDropDown=False)
    dv.showErrorMessage = False
    dv.promptTitle = "Figurenmarkierung"
    dv.prompt = ("O = keine Figur | B-FIGUR = erstes Wort einer Figur | "
                 "I-FIGUR = Fortsetzung")
    ws.add_data_validation(dv)
    dv.add(f"{mk}2:{mk}{letzte}")

    ws.protection.sheet = True
    ws.protection.selectUnlockedCells = False
    ws.protection.selectLockedCells = False
    ws.protection.formatColumns = False
    ws.protection.formatRows = False
    ws.protection.autoFilter = False
    ws.protection.sort = True
    ws.protection.deleteRows = True
    ws.protection.insertRows = True

    leg = wb.create_sheet("Anleitung", 0)
    text = [
        (f"Figurenannotation — {person}, Tranche {TRANCHE}", ""),
        ("", ""),
        ("Aufgabe:", "Markiere die Woerter, die eine Figur bezeichnen."),
        ("", ""),
        ("O", "keine Figur (Voreinstellung, meistens richtig)"),
        ("B-FIGUR", "ERSTES Wort einer Figurenbezeichnung"),
        ("I-FIGUR", "FORTSETZUNG derselben Figurenbezeichnung"),
        ("", ""),
        ("Beispiel:", "Satz: Die alte Hexe sprach"),
        ("  Die", "O"),
        ("  alte", "O"),
        ("  Hexe", "B-FIGUR"),
        ("  sprach", "O"),
        ("", ""),
        ("Satzkontext:", "steht in der Spalte 'satz_text', jeweils in der"),
        ("", "ersten Zeile des Satzes."),
        ("Herkunft:", "Spalten 'sammlung' und 'erzaehlung' nennen das"),
        ("", "Maerchen, falls Sie dort nachlesen moechten."),
        ("", ""),
        ("Unsicher?", "Beste Vermutung eintragen UND x in 'unsicher'."),
        ("", "Gekennzeichnete Unsicherheit hilft, Raten schadet."),
        ("", ""),
        ("Nicht erlaubt:", "sortieren, Zeilen loeschen, Blattschutz aufheben,"),
        ("", "die Tokenspalte veraendern."),
        ("", ""),
        ("Regeln:", "goldstandard_v1.18_Anleitung_annotation_extern.docx"),
    ]
    for i, (a, b) in enumerate(text, start=1):
        ca = leg.cell(row=i, column=1, value=a)
        cb = leg.cell(row=i, column=2, value=b)
        ca.font = Font(name="Arial", bold=(i == 1 or a.endswith(":")))
        cb.font = Font(name="Arial")
    leg.column_dimensions["A"].width = 20
    leg.column_dimensions["B"].width = 58
    leg.protection.sheet = True

    ziel.parent.mkdir(parents=True, exist_ok=True)
    wb.save(ziel)
    return letzte - 1


print("Baufunktion definiert.")

Baufunktion definiert.


## Schritt 4 — Mappen erzeugen und kontrollieren


In [5]:
from openpyxl import load_workbook

print("Arbeitsmappen:")
for person, folge in reihenfolgen.items():
    ziel = BASIS_ARBEIT / f"{person}_ner_tranche{TRANCHE}.xlsx"
    if ziel.exists() and not UEBERSCHREIBEN:
        print(f"  [uebersprungen] {ziel.name}  (existiert bereits)")
        continue
    anzahl = baue_mappe(person, folge, ziel)
    print(f"  [geschrieben  ] {ziel.name}  ({anzahl} Token)")

print()
print("Kontrolle:")
referenz = None
for person in BEARBEITER:
    pfad = BASIS_ARBEIT / f"{person}_ner_tranche{TRANCHE}.xlsx"
    if not pfad.exists():
        print(f"  {person}: Datei fehlt")
        continue
    ws = load_workbook(pfad)["NER"]
    kopf = [c.value for c in ws[1]]
    sp = {n: kopf.index(n) + 1 for n in SPALTEN}
    n = ws.max_row - 1
    ids = {ws.cell(row=i, column=sp["id"]).value for i in range(2, ws.max_row + 1)}
    nicht_o = sum(1 for i in range(2, ws.max_row + 1)
                  if ws.cell(row=i, column=sp["markierung"]).value != "O")
    if referenz is None:
        referenz = ids
    status = ["Tokenzahl ok" if n == t_token else f"ABWEICHUNG {n}",
              "alles O" if nicht_o == 0 else f"WARNUNG: {nicht_o} vorbelegt",
              "IDs eindeutig" if len(ids) == n else "WARNUNG: IDs doppelt",
              "IDs deckungsgleich" if ids == referenz else "WARNUNG: andere Menge"]
    print(f"  {person}: {n} Token | " + " | ".join(status))

print()
print("Ausrichtung fuer Vergleich und Zusammenfuehrung: Spalte 'id'.")

Arbeitsmappen:
  [geschrieben  ] A1_ner_tranche2.xlsx  (2343 Token)
  [geschrieben  ] A3_ner_tranche2.xlsx  (2343 Token)
  [geschrieben  ] A7_ner_tranche2.xlsx  (2343 Token)
  [geschrieben  ] A8_ner_tranche2.xlsx  (2343 Token)

Kontrolle:
  A1: 2343 Token | Tokenzahl ok | alles O | IDs eindeutig | IDs deckungsgleich
  A3: 2343 Token | Tokenzahl ok | alles O | IDs eindeutig | IDs deckungsgleich
  A7: 2343 Token | Tokenzahl ok | alles O | IDs eindeutig | IDs deckungsgleich
  A8: 2343 Token | Tokenzahl ok | alles O | IDs eindeutig | IDs deckungsgleich

Ausrichtung fuer Vergleich und Zusammenfuehrung: Spalte 'id'.
